## 미세먼지 경보 데이터로 머신러닝 실습

> - **분류**: "이 미세먼지 경보, 주의보인가? 경보인가?" → 카테고리 예측
> - **회귀**: "발령 당시 미세먼지 농도가 얼마였을까?" → 숫자 예측

**사용 데이터**: `finedust_2020_2025.csv` (공공데이터 포털)
미세먼지 경보 발령 기록 

## 데이터 소개 & 문제 정의

### 미세먼지 경보 시스템

한국 환경부는 미세먼지 농도가 일정 기준을 넘으면 경보를 발령합니다.

```
PM2.5 (초미세먼지):
  주의보 → 시간 평균 농도 ≥ 75 ㎍/㎥  (2시간 이상 지속)
  경  보 → 시간 평균 농도 ≥ 150 ㎍/㎥  (2시간 이상 지속)

PM10 (미세먼지):
  주의보 → 시간 평균 농도 ≥ 150 ㎍/㎥
  경  보 → 시간 평균 농도 ≥ 300 ㎍/㎥
```

### 데이터 컬럼 설명

| 컬럼 | 설명 | 예시 |
|------|------|------|
| `districtName` | 시·도명 | 서울, 경기, 충북 |
| `dataDate` | 발령일 | 2020-12-29 |
| **`issueVal`** | **발령 당시 농도 (㎍/㎥)** ← **회귀 타깃** | 77 |
| `issueTime` | 발령 시각 | 13:00 |
| **`issueGbn`** | **주의보/경보** ← **분류 타깃** | 주의보 |
| `clearVal` | 해제 당시 농도 (㎍/㎥) | 33 |
| `clearTime` | 해제 시각 | 19:00 |
| `clearDate` | 해제일 | 2020-12-29 |
| `itemCode` | 오염물질 종류 | PM10, PM25 |
| `moveName` | 권역명 (세부 지역) | 중부권역 |
| `sn` | 일련번호 (분석 불필요) | 323 |

### 같은 데이터 → 두 가지 ML 문제

```
        발령일   지역    발령농도    해제농도   오염물질   결과
        ───────────────────────────────────────────────────────
데이터:  2020-12  경기    77 ㎍/㎥   33 ㎍/㎥   PM25      주의보

분류 문제: 이 발령은 주의보인가 경보인가?  →  "주의보"  (범주)
회귀 문제: 발령 당시 농도는 얼마인가?   →  77.0     (숫자)
```

> **핵심**: 분류는 '어느 그룹?', 회귀는 '얼마?'
> 같은 데이터에서 타깃을 무엇으로 정하느냐에 따라 문제가 달라집니다.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 분류 모델
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# 회귀 모델
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

# 평가 지표
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

plt.rc('font', family='Malgun Gothic')
plt.rc('axes', unicode_minus=False)


## 탐색적 데이터 분석 (EDA)

**EDA 순서 예**
1. 데이터 로드 & 기본 정보
2. 결측값 확인
3. 분류 타깃 분포 (주의보 vs 경보)
4. 시도별 발령 현황
5. 연도별·월별 발령 트렌드
6. PM10 vs PM25 비교
7. 발령 농도 분포


In [ ]:
df = pd.read_csv('finedust_2020_2025.csv')
df

In [ ]:
X_train, X_test, y_train, y_test = train_test_split( X, y,
            test_size=0.2,
            random_state=0,
            stratify=y
) #stratify=y는 train_test_split()에서 주의보/경보 비율을 유지하면서 데이터를 나누는 옵션, 분류에서 사용

- sklearn은 가나다 순(사전순) 으로 인코딩합니다. 따라서 0 = 경보 1 = 주의보

## 회귀 (Regression) — 발령 농도(issueVal) 예측

> **특성에서 issueVal 제외**     
> 회귀의 타깃이 `issueVal`이므로, 특성으로 `issueVal`을 쓰면       
> "답을 보고 시험 치는 것" (데이터 누수, Data Leakage)!       
> `clearVal`(해제 농도)은 발령 이후 정보지만 교육 목적으로 사용합니다    

### → 더 좋은 모델 만들려면: 기온, 풍속, 습도 등 기상 데이터 추가 필요!